# 02b — Evaluación comparativa de estrategias de chunking

**Proyecto:** Chem RAG Assistant  
**Repositorio:** https://github.com/Jesusrodriguezf90/chem-rag-assistant  
**Fase:** Evaluación y optimización del pipeline

---

Este notebook evalúa y compara cuatro estrategias de chunking sobre el mismo
documento científico y el mismo conjunto de preguntas con ground truth definido
en `03_recuperacion.ipynb`, siguiendo el estándar profesional de evaluación RAG.

**Entrada:** `data/processed/PMC10967698_extracted.md` (generado en `01_ingesta.ipynb`)  
**Salida:** tabla comparativa de métricas de recuperación por estrategia

### Estrategias evaluadas

| # | Estrategia | Descripción resumida |
|---|---|---|
| 1 | `RecursiveCharacterTextSplitter` | **Baseline.** Divide respetando párrafos, frases y palabras en ese orden. Es el estándar de facto en producción por su equilibrio entre simplicidad y calidad. |
| 2 | `SemanticChunker` | Divide en función de la similitud semántica entre frases consecutivas. Crea chunks más coherentes temáticamente pero con mayor coste computacional. |
| 3 | `MarkdownHeaderTextSplitter` | Divide respetando los encabezados `##` y `###` del documento. Ideal para papers científicos con estructura bien definida por secciones. |
| 4 | Contextual Chunk Headers | Extiende la Estrategia 1 anteponiendo el título de sección a cada chunk. Preserva el contexto estructural sin coste de API adicional. |

### ¿Por qué comparar estrategias?

El chunking es el paso que más impacta la calidad de recuperación en un pipeline RAG.
Una estrategia incorrecta puede crear vectores sin contexto semántico suficiente,
o fragmentar información relacionada que debería recuperarse junta. La evaluación
comparativa con métricas cuantitativas es el único método riguroso para elegir
la estrategia óptima para un tipo de documento concreto.

In [ ]:
"""
Notebook: 02b_chunking_eval.ipynb

Objetivo:
    Comparar cuatro estrategias de chunking sobre el documento científico
    PMC10967698 usando las mismas preguntas de evaluación con ground truth
    definidas en 03_recuperacion.ipynb. Todas las estrategias se evalúan
    en condiciones idénticas (mismo modelo de embeddings, misma base vectorial,
    mismo top-k) para garantizar comparabilidad.

    Estrategias evaluadas:
      1. RecursiveCharacterTextSplitter — baseline de producción
      2. SemanticChunker — chunking por similitud semántica
      3. MarkdownHeaderTextSplitter — chunking por estructura del documento
      4. Contextual Chunk Headers — baseline + contexto de sección prepend

    Este notebook cubre:
      1.  Configuración del entorno
      2.  Instalación de dependencias
      3.  Definición de rutas, parámetros y preguntas de evaluación
      4.  Carga del documento procesado
      5.  Implementación de las 4 estrategias de chunking
      6.  Carga del modelo de embeddings (BGE-M3)
      7.  Función de indexación y recuperación reutilizable
      8.  Evaluación de cada estrategia con métricas cuantitativas
      9.  Comparativa final y selección de la estrategia óptima
      10. Resumen de resultados y recomendación

Fuente de datos:
    Büchele WRE, Schlachta TP, Gebendorfer AL, Pamperin J, Richter LF,
    Sauer MJ, Prokop A, Kühn FE. Synthesis, characterization, and biomedical
    evaluation of ethylene-bridged tetra-NHC Pd(ii), Pt(ii) and Au(iii)
    complexes, with apoptosis-inducing properties in cisplatin-resistant
    neuroblastoma cells. Frontiers in Chemistry. 2024.
    PMC: https://pmc.ncbi.nlm.nih.gov/articles/PMC10967698/

    Documento utilizado exclusivamente con fines de investigación y desarrollo.
    No se distribuye ni se incluye en el repositorio.

Autor:   Jesús Rodríguez
Fecha:   2026-05-04
Versión: 1.0.0
"""

## 1. Configuración del entorno

In [1]:
# Detección del entorno de ejecución (Colab vs local)
try:
    from google.colab import drive
    IN_COLAB = True
    print("Entorno detectado: Google Colab")
except ImportError:
    IN_COLAB = False
    print("Entorno detectado: local")

Entorno detectado: Google Colab


In [2]:
# Montaje de Google Drive (solo en Colab)
if IN_COLAB:
    drive.mount('/content/drive')
    print("Google Drive montado correctamente")

Mounted at /content/drive
Google Drive montado correctamente


In [3]:
# Verificación del entorno de ejecución
import sys
import platform

print(f"Python : {sys.version}")
print(f"Sistema: {platform.system()} {platform.release()}")

# Detección de GPU — relevante para SemanticChunker y BGE-M3
import torch

if torch.cuda.is_available():
    dispositivo = "cuda"
    nombre_gpu = torch.cuda.get_device_name(0)
    print(f"Dispositivo : GPU — {nombre_gpu}")
else:
    dispositivo = "cpu"
    print("Dispositivo : CPU (correcto para prototipado)")

Python : 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Sistema: Linux 6.6.113+
Dispositivo : CPU (correcto para prototipado)


## 2. Instalación de dependencias

In [4]:
# Instalación de todas las dependencias necesarias para las 4 estrategias
# sentence-transformers: declarado explícitamente aunque llegue como
# dependencia transitiva de FlagEmbedding — buena práctica de reproducibilidad
# langchain-experimental: contiene SemanticChunker
%pip install langchain langchain-text-splitters langchain-experimental \
             sentence-transformers FlagEmbedding chromadb -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.1/210.1 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 82.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 78.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 78.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231

In [5]:
# Verificación de imports críticos para las 4 estrategias
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    MarkdownHeaderTextSplitter,
)
from langchain_experimental.text_splitter import SemanticChunker

print("Imports de estrategias de chunking: OK")

Imports de estrategias de chunking: OK


## 3. Definición de rutas, parámetros y preguntas de evaluación

In [6]:
# Librería estándar para manejo de rutas
from pathlib import Path

# Ruta raíz del proyecto en Google Drive
PROYECTO_RAIZ = Path('/content/drive/MyDrive/chem-rag-assistant')

# Documento procesado y limpio — output de 01_ingesta.ipynb
RUTA_MD = PROYECTO_RAIZ / 'data' / 'processed' / 'PMC10967698_extracted.md'

# Directorio de salida para artefactos de evaluación
DIR_EVAL = PROYECTO_RAIZ / 'data' / 'eval'
DIR_EVAL.mkdir(parents=True, exist_ok=True)

# Directorio temporal para ChromaDB — se recrea por estrategia
DIR_CHROMA_EVAL = PROYECTO_RAIZ / 'data' / 'chroma_eval'
DIR_CHROMA_EVAL.mkdir(parents=True, exist_ok=True)

# Parámetros de evaluación — idénticos a los usados en 03_recuperacion
# para garantizar comparabilidad entre notebooks
TOP_K             = 4
MODELO_EMBEDDINGS = 'BAAI/bge-m3'

# Parámetros de chunking para estrategias basadas en tamaño
CHUNK_SIZE    = 1000
CHUNK_OVERLAP = 100

# Longitud mínima de chunk — chunks por debajo se fusionan con el siguiente
# (misma lógica aplicada en 02_embeddings.ipynb)
LONGITUD_MINIMA = 100

# Validación de existencia del documento de entrada
assert RUTA_MD.exists(), (
    f"Documento no encontrado: {RUTA_MD}\n"
    f"Ejecuta primero el notebook 01_ingesta.ipynb"
)

print(f"Documento de entrada  : {RUTA_MD.name}")
print(f"Directorio evaluación : {DIR_EVAL}")
print(f"Top-k                 : {TOP_K}")
print(f"Chunk size            : {CHUNK_SIZE} caracteres")
print(f"Chunk overlap         : {CHUNK_OVERLAP} caracteres")

Documento de entrada  : PMC10967698_extracted.md
Directorio evaluación : /content/drive/MyDrive/chem-rag-assistant/data/eval
Top-k                 : 4
Chunk size            : 1000 caracteres
Chunk overlap         : 100 caracteres


In [7]:
# Preguntas de evaluación con ground truth
# Idénticas a las definidas en 03_recuperacion.ipynb para garantizar
# que la comparación entre estrategias sea directamente comparable
# con los resultados de recuperación del pipeline base
PREGUNTAS_EVALUACION = [
    {
        "pregunta"       : "What metals are used in the NHC complexes studied?",
        "nivel"          : "básica",
        "ground_truth"   : "Palladium (Pd), platinum (Pt) and gold (Au).",
        "chunk_esperado" : "Introduction or Synthesis section mentioning Pd, Pt, Au",
    },
    {
        "pregunta"       : (
            "What is the effect of the complexes on "
            "cisplatin-resistant neuroblastoma cells?"
        ),
        "nivel"          : "básica",
        "ground_truth"   : (
            "The complexes induce apoptosis in cisplatin-resistant "
            "SK-N-AS neuroblastoma cells."
        ),
        "chunk_esperado" : "Biological evaluation section",
    },
    {
        "pregunta"       : (
            "What analytical techniques were used "
            "to characterize the compounds?"
        ),
        "nivel"          : "básica",
        "ground_truth"   : (
            "NMR spectroscopy (1H, 13C, 31P), ESI mass spectrometry "
            "and IR spectroscopy."
        ),
        "chunk_esperado" : "General procedures and analytical methods section",
    },
    {
        "pregunta"       : (
            "What is the role of the ethylene bridge "
            "in the tetra-NHC ligand design?"
        ),
        "nivel"          : "intermedia",
        "ground_truth"   : (
            "The ethylene bridge connects two NHC units forming a "
            "tetradentate chelating ligand that stabilizes the metal center."
        ),
        "chunk_esperado" : "Introduction section on NHC ligand design",
    },
    {
        "pregunta"       : (
            "How do the cytotoxicity results of the Au(III) "
            "complexes compare to cisplatin?"
        ),
        "nivel"          : "avanzada",
        "ground_truth"   : (
            "The Au(III) complexes show higher cytotoxicity than cisplatin "
            "in resistant cell lines, overcoming cisplatin resistance."
        ),
        "chunk_esperado" : "Biological evaluation section on cytotoxicity results",
    },
]

print(f"Preguntas de evaluación cargadas: {len(PREGUNTAS_EVALUACION)}")
print("-" * 45)
for item in PREGUNTAS_EVALUACION:
    print(f"  [{item['nivel'].upper()}] {item['pregunta'][:55]}...")

Preguntas de evaluación cargadas: 5
---------------------------------------------
  [BÁSICA] What metals are used in the NHC complexes studied?...
  [BÁSICA] What is the effect of the complexes on cisplatin-resist...
  [BÁSICA] What analytical techniques were used to characterize th...
  [INTERMEDIA] What is the role of the ethylene bridge in the tetra-NH...
  [AVANZADA] How do the cytotoxicity results of the Au(III) complexe...


## 4. Carga del documento procesado

In [8]:
# Lectura del documento limpio generado en 01_ingesta.ipynb
# Este archivo ya contiene el texto procesado con limpieza universal
# y específica — es la entrada correcta para todas las estrategias
with open(RUTA_MD, 'r', encoding='utf-8') as f:
    documento_md = f.read()

print(f"Documento cargado correctamente")
print(f"Caracteres totales    : {len(documento_md):,}")
print(f"Palabras aproximadas  : {len(documento_md.split()):,}")

Documento cargado correctamente
Caracteres totales    : 49,922
Palabras aproximadas  : 8,883


## 5. Implementación de las 4 estrategias de chunking

Cada estrategia incluye la misma lógica de fusión de chunks cortos
que validamos en `02_embeddings.ipynb` para garantizar comparabilidad.

In [35]:
# Función auxiliar reutilizable para fusionar chunks cortos
# Misma lógica que en 02_embeddings.ipynb — los títulos de sección
# aislados se fusionan con el chunk siguiente para preservar contexto
def fusionar_chunks_cortos(
    chunks: list[str],
    longitud_minima: int = LONGITUD_MINIMA,
) -> list[str]:
    """Fusiona chunks por debajo de la longitud mínima con el siguiente.
    También elimina chunks vacíos o de solo espacios en blanco que
    puedan generarse durante el splitting semántico o por limpieza
    del documento.
    """
    # Eliminar chunks vacíos antes de procesar
    # para evitar contaminación del índice vectorial
    chunks_no_vacios = [c for c in chunks if c.strip()]

    chunks_fusionados = []
    i = 0
    while i < len(chunks_no_vacios):
        if (
            len(chunks_no_vacios[i]) < longitud_minima
            and i + 1 < len(chunks_no_vacios)
        ):
            chunks_fusionados.append(
                chunks_no_vacios[i] + "\n" + chunks_no_vacios[i + 1]
            )
            i += 2
        else:
            chunks_fusionados.append(chunks_no_vacios[i])
            i += 1
    return chunks_fusionados

In [10]:
# ================================================================
# ESTRATEGIA 1 — RecursiveCharacterTextSplitter (baseline)
# ================================================================
# Divide el texto respetando párrafos, frases y palabras en ese orden
# de prioridad. Es el método estándar de facto en producción por su
# equilibrio entre simplicidad, velocidad y calidad de recuperación.
# Los separadores Markdown (## y ###) se incluyen para respetar
# la estructura del paper científico.
splitter_1 = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n## ", "\n### ", "\n\n", "\n", " "],
)

chunks_raw_1 = splitter_1.split_text(documento_md)
chunks_1     = fusionar_chunks_cortos(chunks_raw_1)

print("Estrategia 1 — RecursiveCharacterTextSplitter")
print(f"  Chunks originales  : {len(chunks_raw_1)}")
print(f"  Chunks tras fusión : {len(chunks_1)}")
print(
    f"  Tamaño medio       : "
    f"{sum(len(c) for c in chunks_1) // len(chunks_1)} caracteres"
)

Estrategia 1 — RecursiveCharacterTextSplitter
  Chunks originales  : 87
  Chunks tras fusión : 77
  Tamaño medio       : 678 caracteres


In [11]:
# ================================================================
# ESTRATEGIA 3 — MarkdownHeaderTextSplitter
# ================================================================
# Divide el documento respetando los encabezados ## y ### como
# fronteras naturales de sección. Ideal para documentos científicos
# con estructura bien definida porque garantiza que cada chunk
# contiene exactamente el contenido de una sección del paper.
# Se ejecuta antes de la Estrategia 2 porque no requiere GPU.
splitter_3 = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("##", "seccion"),
        ("###", "subseccion"),
    ],
    strip_headers=False,  # Conservar el título de sección en el chunk
)

# MarkdownHeaderTextSplitter devuelve objetos Document, no strings
# Extraemos el contenido de página y añadimos el encabezado como prefijo
docs_3   = splitter_3.split_text(documento_md)
chunks_3 = [
    doc.page_content
    for doc in docs_3
    if len(doc.page_content.strip()) >= LONGITUD_MINIMA
]

print("Estrategia 3 — MarkdownHeaderTextSplitter")
print(f"  Chunks generados   : {len(chunks_3)}")
print(
    f"  Tamaño medio       : "
    f"{sum(len(c) for c in chunks_3) // len(chunks_3)} caracteres"
)
print(f"  Chunk más corto    : {min(len(c) for c in chunks_3)} caracteres")
print(f"  Chunk más largo    : {max(len(c) for c in chunks_3)} caracteres")

Estrategia 3 — MarkdownHeaderTextSplitter
  Chunks generados   : 17
  Tamaño medio       : 2931 caracteres
  Chunk más corto    : 256 caracteres
  Chunk más largo    : 14202 caracteres


In [12]:
# ================================================================
# ESTRATEGIA 4 — Contextual Chunk Headers
# ================================================================
# Extiende la Estrategia 1 (baseline) anteponiendo el título de sección
# Markdown al inicio de cada chunk. Esto preserva el contexto
# estructural del documento sin llamadas adicionales a ninguna API,
# resolviendo el problema de chunks que pierden su sección de origen
# tras el splitting. Implementación local de Anthropic Contextual Retrieval.
import re

def extraer_seccion_activa(texto: str, documento_completo: str) -> str:
    """Extrae el título de sección (##) más reciente antes del chunk."""
    pos = documento_completo.find(texto[:100])
    if pos == -1:
        return ""
    fragmento_previo = documento_completo[:pos]
    secciones = re.findall(r'^##[^#].*$', fragmento_previo, re.MULTILINE)
    return secciones[-1].strip() if secciones else ""


# Generar chunks base con RecursiveCharacterTextSplitter
chunks_raw_4 = splitter_1.split_text(documento_md)
chunks_base_4 = fusionar_chunks_cortos(chunks_raw_4)

# Añadir contexto de sección a cada chunk
chunks_4 = []
for chunk in chunks_base_4:
    seccion = extraer_seccion_activa(chunk, documento_md)
    if seccion and not chunk.startswith(seccion):
        # Anteposición del título de sección solo si el chunk no lo contiene ya
        chunks_4.append(f"{seccion}\n\n{chunk}")
    else:
        chunks_4.append(chunk)

print("Estrategia 4 — Contextual Chunk Headers")
print(f"  Chunks generados   : {len(chunks_4)}")
print(
    f"  Tamaño medio       : "
    f"{sum(len(c) for c in chunks_4) // len(chunks_4)} caracteres"
)
# Verificar que el contexto se añadió correctamente
chunks_con_contexto = sum(
    1 for c in chunks_4 if c.startswith("##")
)
print(f"  Chunks con contexto: {chunks_con_contexto} de {len(chunks_4)}")

Estrategia 4 — Contextual Chunk Headers
  Chunks generados   : 77
  Tamaño medio       : 719 caracteres
  Chunks con contexto: 77 de 77


## 6. Carga del modelo de embeddings (BGE-M3)

In [13]:
# Carga de BGE-M3 — modelo compartido por todas las estrategias
# Es crítico usar el mismo modelo para garantizar que los vectores
# de todas las estrategias están en el mismo espacio semántico
# y la comparación es justa
from FlagEmbedding import BGEM3FlagModel

print(f"Cargando modelo {MODELO_EMBEDDINGS}...")
print("(La primera ejecución descarga ~2.2 GB — puede tardar varios minutos)")

modelo = BGEM3FlagModel(
    MODELO_EMBEDDINGS,
    use_fp16=True,      # Reduce uso de memoria a la mitad sin pérdida significativa
    device=dispositivo,
)

print(f"Modelo cargado en: {dispositivo.upper()}")

Cargando modelo BAAI/bge-m3...
(La primera ejecución descarga ~2.2 GB — puede tardar varios minutos)


config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Modelo cargado en: CPU


## 7. Estrategia 2 — SemanticChunker

In [16]:
# ================================================================
# ESTRATEGIA 2 — SemanticChunker
# ================================================================
# Divide el texto calculando la similitud semántica entre frases
# consecutivas. Cuando la similitud cae por debajo de un umbral,
# se considera un cambio de tema y se crea un nuevo chunk.
# Esto produce chunks temáticamente más coherentes que el baseline,
# a costa de mayor tiempo de procesamiento (requiere embeddings
# de todas las frases antes de dividir).
#
# Se ejecuta después de cargar BGE-M3 porque el SemanticChunker
# necesita un modelo de embeddings para calcular similitudes.
# Usamos el mismo BGE-M3 para mantener consistencia con el pipeline.
%pip install langchain-huggingface -q

from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings

# Wrapper de HuggingFace necesario para que LangChain use BGE-M3
# como modelo de embeddings en el SemanticChunker
embeddings_lc = HuggingFaceEmbeddings(
    model_name=MODELO_EMBEDDINGS,
    model_kwargs={'device': dispositivo},
    encode_kwargs={'normalize_embeddings': True},
)

# breakpoint_threshold_type='percentile' divide cuando la similitud
# entre frases cae en el percentil más bajo — más sensible a cambios
# de tema que el umbral fijo, adecuado para papers científicos
# con secciones temáticamente distintas
splitter_2 = SemanticChunker(
    embeddings=embeddings_lc,
    breakpoint_threshold_type='percentile',
    breakpoint_threshold_amount=85,
)

print("Generando chunks semánticos (puede tardar varios minutos en CPU)...")
chunks_raw_2 = splitter_2.split_text(documento_md)
chunks_2     = fusionar_chunks_cortos(chunks_raw_2)

print("Estrategia 2 — SemanticChunker")
print(f"  Chunks originales  : {len(chunks_raw_2)}")
print(f"  Chunks tras fusión : {len(chunks_2)}")
print(
    f"  Tamaño medio       : "
    f"{sum(len(c) for c in chunks_2) // len(chunks_2)} caracteres"
)
print(f"  Chunk más corto    : {min(len(c) for c in chunks_2)} caracteres")
print(f"  Chunk más largo    : {max(len(c) for c in chunks_2)} caracteres")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Generando chunks semánticos (puede tardar varios minutos en CPU)...
Estrategia 2 — SemanticChunker
  Chunks originales  : 67
  Chunks tras fusión : 57
  Tamaño medio       : 874 caracteres
  Chunk más corto    : 0 caracteres
  Chunk más largo    : 7159 caracteres


## 8. Función de indexación y recuperación reutilizable

Para garantizar que la comparación entre estrategias es justa, se usa
una única función que indexa y recupera de forma idéntica para todas.
La única variable entre estrategias es la lista de chunks de entrada.

In [23]:
# Función de generación de embeddings reutilizable
# Encapsula la llamada a BGE-M3 para evitar repetición de código
# en cada estrategia
import numpy as np

def generar_embeddings(chunks: list[str]) -> np.ndarray:
    """Genera embeddings densos para una lista de chunks con BGE-M3."""
    resultado = modelo.encode(
        chunks,
        batch_size=4,        # Conservador para CPU
        max_length=512,      # Suficiente para nuestros chunks
        return_dense=True,
        return_sparse=False,
        return_colbert_vecs=False,
    )
    return resultado['dense_vecs']


print("Función generar_embeddings definida correctamente")

Función generar_embeddings definida correctamente


In [24]:
# Función de indexación en ChromaDB reutilizable
# Crea una colección nueva por estrategia para evitar contaminación
# entre evaluaciones
import chromadb

def indexar_chunks(
    chunks: list[str],
    embeddings: np.ndarray,
    nombre_coleccion: str,
) -> chromadb.Collection:
    """Indexa chunks y embeddings en una colección ChromaDB nueva."""
    cliente = chromadb.PersistentClient(path=str(DIR_CHROMA_EVAL))

    # Eliminar colección previa si existe para evitar duplicados
    try:
        cliente.delete_collection(name=nombre_coleccion)
    except Exception:
        pass

    # Crear colección con métrica coseno — correcta para BGE-M3
    coleccion = cliente.create_collection(
        name=nombre_coleccion,
        metadata={"hnsw:space": "cosine"},
    )

    coleccion.add(
        ids=[f"chunk_{i:04d}" for i in range(len(chunks))],
        embeddings=embeddings.tolist(),
        documents=chunks,
        metadatas=[
            {
                "fuente"  : "PMC10967698",
                "indice"  : i,
                "longitud": len(chunks[i]),
            }
            for i in range(len(chunks))
        ],
    )

    return coleccion


print("Función indexar_chunks definida correctamente")

Función indexar_chunks definida correctamente


In [25]:
# Función de recuperación reutilizable
# Idéntica a la usada en 03_recuperacion.ipynb para garantizar
# que las métricas son directamente comparables entre notebooks
#
# NOTA PARA MIGRACIÓN A src/retrieval/vector_store.py:
# 1. Añadir try/except con logging para trazabilidad en producción
# 2. Añadir filtrado por metadatos (fuente=) para soporte multi-documento
def recuperar_chunks(
    pregunta: str,
    coleccion: chromadb.Collection,
    top_k: int = TOP_K,
) -> dict:
    """Recupera los top-k chunks más similares a la pregunta."""
    resultado = modelo.encode(
        [pregunta],
        batch_size=1,
        max_length=512,
        return_dense=True,
        return_sparse=False,
        return_colbert_vecs=False,
    )
    vector_consulta = resultado['dense_vecs'][0].tolist()

    return coleccion.query(
        query_embeddings=[vector_consulta],
        n_results=top_k,
        include=["documents", "distances"],
    )


print("Función recuperar_chunks definida correctamente")

Función recuperar_chunks definida correctamente


In [26]:
# Función de evaluación de una estrategia completa
# Ejecuta el ciclo completo: embeddings → indexación → recuperación → métricas
# para un conjunto de chunks y devuelve los resultados estructurados
def evaluar_estrategia(
    nombre: str,
    chunks: list[str],
    preguntas: list[dict],
) -> dict:
    """Evalúa una estrategia de chunking completa.

    Args:
        nombre: identificador de la estrategia (usado como nombre de colección)
        chunks: lista de chunks generados por la estrategia
        preguntas: lista de preguntas con ground truth

    Returns:
        Diccionario con métricas de recuperación por pregunta y totales
    """
    print(f"\nEvaluando estrategia: {nombre}")

    # Filtrar chunks inválidos (vacíos o solo espacios en blanco)
    # antes de indexar para evitar errores en ChromaDB.
    # Pueden generarse en SemanticChunker cuando detecta un cambio
    # semántico en un punto sin contenido real (saltos de línea,
    # separadores vacíos tras la limpieza del documento)
    chunks_validos = [c for c in chunks if c.strip()]
    n_filtrados    = len(chunks) - len(chunks_validos)

    if n_filtrados > 0:
        print(
            f"  [AVISO] {n_filtrados} chunk(s) vacío(s) eliminado(s) "
            f"antes de indexar"
        )

    print(f"  Chunks indexados  : {len(chunks_validos)}")

    # Generar embeddings
    print("  Generando embeddings...")
    embeddings = generar_embeddings(chunks_validos)

    # Indexar en ChromaDB
    nombre_coleccion = f"eval_{nombre.lower().replace(' ', '_')}"
    coleccion        = indexar_chunks(chunks_validos, embeddings, nombre_coleccion)

    # Evaluar cada pregunta
    resultados_preguntas = []

    for item in preguntas:
        pregunta     = item['pregunta']
        nivel        = item['nivel']
        ground_truth = item['ground_truth']

        resultado = recuperar_chunks(pregunta, coleccion)

        distancias  = resultado['distances'][0]
        # ChromaDB con métrica coseno devuelve distancias (1 - similitud)
        similitudes = [round(1 - d, 4) for d in distancias]

        # Métricas de recuperación
        sim_top1  = similitudes[0]
        sim_media = round(sum(similitudes) / len(similitudes), 4)

        resultados_preguntas.append({
            "pregunta"    : pregunta,
            "nivel"       : nivel,
            "ground_truth": ground_truth,
            "sim_top1"    : sim_top1,
            "sim_media"   : sim_media,
            "similitudes" : similitudes,
        })

    # Métricas globales de la estrategia
    sim_top1_global  = round(
        sum(r['sim_top1'] for r in resultados_preguntas)
        / len(resultados_preguntas), 4
    )
    sim_media_global = round(
        sum(r['sim_media'] for r in resultados_preguntas)
        / len(resultados_preguntas), 4
    )

    print(f"  Sim. top-1 media  : {sim_top1_global:.4f}")
    print(f"  Sim. media global : {sim_media_global:.4f}")

    return {
        "nombre"          : nombre,
        "n_chunks"        : len(chunks_validos),
        "sim_top1_global" : sim_top1_global,
        "sim_media_global": sim_media_global,
        "resultados"      : resultados_preguntas,
    }


print("Función evaluar_estrategia definida correctamente")

Función evaluar_estrategia definida correctamente


In [27]:
# Evaluación de las 4 estrategias
resultado_1 = evaluar_estrategia(
    nombre="estrategia_1_recursive",
    chunks=chunks_1,
    preguntas=PREGUNTAS_EVALUACION,
)


Evaluando estrategia: estrategia_1_recursive
  Chunks indexados  : 77
  Generando embeddings...


Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  2.17it/s]

  Sim. top-1 media  : 0.5719
  Sim. media global : 0.5463


In [28]:
resultado_2 = evaluar_estrategia(
    nombre="estrategia_2_semantic",
    chunks=chunks_2,
    preguntas=PREGUNTAS_EVALUACION,
)


Evaluando estrategia: estrategia_2_semantic
  [AVISO] 1 chunk(s) vacío(s) eliminado(s) antes de indexar
  Chunks indexados  : 56
  Generando embeddings...


Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  2.04it/s]

  Sim. top-1 media  : 0.5692
  Sim. media global : 0.5375


In [29]:
resultado_3 = evaluar_estrategia(
    nombre="estrategia_3_markdown",
    chunks=chunks_3,
    preguntas=PREGUNTAS_EVALUACION,
)


Evaluando estrategia: estrategia_3_markdown
  Chunks indexados  : 17
  Generando embeddings...


Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  2.04it/s]

  Sim. top-1 media  : 0.5787
  Sim. media global : 0.5101


In [30]:
resultado_4 = evaluar_estrategia(
    nombre="estrategia_4_contextual",
    chunks=chunks_4,
    preguntas=PREGUNTAS_EVALUACION,
)


Evaluando estrategia: estrategia_4_contextual
  Chunks indexados  : 77
  Generando embeddings...


Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  2.06it/s]

  Sim. top-1 media  : 0.5850
  Sim. media global : 0.5548


## 9. Comparativa final entre estrategias

In [31]:
# Tabla comparativa de todas las estrategias evaluadas
# Incluye métricas globales y desglose por nivel de pregunta
# para identificar en qué tipo de consultas cada estrategia
# tiene ventaja o desventaja
todos_resultados = [resultado_1, resultado_2, resultado_3, resultado_4]

print("=" * 70)
print("COMPARATIVA FINAL — ESTRATEGIAS DE CHUNKING")
print("=" * 70)
print(
    f"  {'Estrategia':<35} {'Chunks':>6} "
    f"{'Sim. top-1':>10} {'Sim. media':>10}"
)
print(f"  {'-' * 65}")

for r in todos_resultados:
    print(
        f"  {r['nombre']:<35} {r['n_chunks']:>6} "
        f"{r['sim_top1_global']:>10.4f} "
        f"{r['sim_media_global']:>10.4f}"
    )

print(f"  {'-' * 65}")

# Identificar la mejor estrategia en cada métrica
mejor_top1  = max(todos_resultados, key=lambda x: x['sim_top1_global'])
mejor_media = max(todos_resultados, key=lambda x: x['sim_media_global'])

print(f"\n  Mejor sim. top-1  : {mejor_top1['nombre']} ({mejor_top1['sim_top1_global']:.4f})")
print(f"  Mejor sim. media  : {mejor_media['nombre']} ({mejor_media['sim_media_global']:.4f})")
print("=" * 70)

COMPARATIVA FINAL — ESTRATEGIAS DE CHUNKING
  Estrategia                          Chunks Sim. top-1 Sim. media
  -----------------------------------------------------------------
  estrategia_1_recursive                  77     0.5719     0.5463
  estrategia_2_semantic                   56     0.5692     0.5375
  estrategia_3_markdown                   17     0.5787     0.5101
  estrategia_4_contextual                 77     0.5850     0.5548
  -----------------------------------------------------------------

  Mejor sim. top-1  : estrategia_4_contextual (0.5850)
  Mejor sim. media  : estrategia_4_contextual (0.5548)


In [32]:
# Desglose por nivel de pregunta para cada estrategia
# Permite identificar si una estrategia es mejor para preguntas
# básicas, intermedias o avanzadas — información relevante para
# decidir si usar estrategias distintas según el tipo de consulta
print("=" * 70)
print("DESGLOSE POR NIVEL DE PREGUNTA")
print("=" * 70)

niveles = ['básica', 'intermedia', 'avanzada']

for nivel in niveles:
    print(f"\n  [{nivel.upper()}]")
    print(
        f"  {'Estrategia':<35} {'Sim. top-1':>10} {'Sim. media':>10}"
    )
    print(f"  {'-' * 57}")

    for r in todos_resultados:
        preguntas_nivel = [
            p for p in r['resultados']
            if p['nivel'] == nivel
        ]
        if preguntas_nivel:
            sim_top1_nivel  = round(
                sum(p['sim_top1'] for p in preguntas_nivel)
                / len(preguntas_nivel), 4
            )
            sim_media_nivel = round(
                sum(p['sim_media'] for p in preguntas_nivel)
                / len(preguntas_nivel), 4
            )
            print(
                f"  {r['nombre']:<35} "
                f"{sim_top1_nivel:>10.4f} "
                f"{sim_media_nivel:>10.4f}"
            )

print("=" * 70)

DESGLOSE POR NIVEL DE PREGUNTA

  [BÁSICA]
  Estrategia                          Sim. top-1 Sim. media
  ---------------------------------------------------------
  estrategia_1_recursive                  0.5791     0.5519
  estrategia_2_semantic                   0.5745     0.5425
  estrategia_3_markdown                   0.5878     0.5177
  estrategia_4_contextual                 0.5917     0.5612

  [INTERMEDIA]
  Estrategia                          Sim. top-1 Sim. media
  ---------------------------------------------------------
  estrategia_1_recursive                  0.5667     0.5524
  estrategia_2_semantic                   0.5629     0.5348
  estrategia_3_markdown                   0.5950     0.5005
  estrategia_4_contextual                 0.5737     0.5582

  [AVANZADA]
  Estrategia                          Sim. top-1 Sim. media
  ---------------------------------------------------------
  estrategia_1_recursive                  0.5555     0.5233
  estrategia_2_semantic    

## 10. Resumen de resultados y recomendación

In [33]:
# Selección de la estrategia óptima basada en las métricas obtenidas
# Criterio de selección: estrategia con mejor combinación de
# sim. top-1 (precisión del chunk más relevante) y sim. media
# (calidad global del contexto enviado al LLM)
#
# La estrategia seleccionada aquí debe reemplazar la Estrategia 1
# en 02_embeddings.ipynb y 03_recuperacion.ipynb si mejora
# significativamente las métricas (diferencia > 0.01)

# Puntuación combinada: media ponderada de sim_top1 (60%) y sim_media (40%)
# Mayor peso a sim_top1 porque el chunk más relevante es el que
# más impacta la calidad de la respuesta generada
for r in todos_resultados:
    r['puntuacion'] = round(
        0.6 * r['sim_top1_global'] + 0.4 * r['sim_media_global'], 4
    )

estrategia_optima = max(todos_resultados, key=lambda x: x['puntuacion'])

print("=" * 70)
print("RESUMEN — EVALUACIÓN COMPARATIVA DE CHUNKING")
print("=" * 70)
print(f"  Modelo embeddings   : {MODELO_EMBEDDINGS}")
print(f"  Top-k               : {TOP_K}")
print(f"  Preguntas evaluadas : {len(PREGUNTAS_EVALUACION)}")
print(f"  Estrategias         : {len(todos_resultados)}")
print()
print(
    f"  {'Estrategia':<35} {'Chunks':>6} "
    f"{'Top-1':>7} {'Media':>7} {'Puntuación':>10}"
)
print(f"  {'-' * 68}")

for r in sorted(todos_resultados, key=lambda x: x['puntuacion'], reverse=True):
    marca = " ◀ ÓPTIMA" if r['nombre'] == estrategia_optima['nombre'] else ""
    print(
        f"  {r['nombre']:<35} {r['n_chunks']:>6} "
        f"{r['sim_top1_global']:>7.4f} "
        f"{r['sim_media_global']:>7.4f} "
        f"{r['puntuacion']:>10.4f}{marca}"
    )

print(f"  {'-' * 68}")
print()
print(f"  Estrategia óptima   : {estrategia_optima['nombre']}")
print(f"  Puntuación          : {estrategia_optima['puntuacion']:.4f}")
print()
print("  MEJORAS FUTURAS IDENTIFICADAS:")
print("  1. Búsqueda híbrida (BM25 + semántica) con Reciprocal Rank Fusion")
print("  2. Reranking con modelo cross-encoder (ej. BAAI/bge-reranker-v2-m3)")
print("  3. Subdivisión de chunks largos en Estrategia 3 (Markdown)")
print("  4. Limpieza de residuos de afiliaciones en documento procesado")
print("  5. Soporte multi-documento con filtrado por metadatos")
print("=" * 70)

RESUMEN — EVALUACIÓN COMPARATIVA DE CHUNKING
  Modelo embeddings   : BAAI/bge-m3
  Top-k               : 4
  Preguntas evaluadas : 5
  Estrategias         : 4

  Estrategia                          Chunks   Top-1   Media Puntuación
  --------------------------------------------------------------------
  estrategia_4_contextual                 77  0.5850  0.5548     0.5729 ◀ ÓPTIMA
  estrategia_1_recursive                  77  0.5719  0.5463     0.5617
  estrategia_2_semantic                   56  0.5692  0.5375     0.5565
  estrategia_3_markdown                   17  0.5787  0.5101     0.5513
  --------------------------------------------------------------------

  Estrategia óptima   : estrategia_4_contextual
  Puntuación          : 0.5729

  MEJORAS FUTURAS IDENTIFICADAS:
  1. Búsqueda híbrida (BM25 + semántica) con Reciprocal Rank Fusion
  2. Reranking con modelo cross-encoder (ej. BAAI/bge-reranker-v2-m3)
  3. Subdivisión de chunks largos en Estrategia 3 (Markdown)
  4. Limpieza de

In [34]:
# Persistencia de los resultados de evaluación en formato JSON
# para referencia futura y comparación con mejoras del pipeline
import json
from datetime import datetime

RUTA_RESULTADOS = DIR_EVAL / 'chunking_eval_results.json'

# Preparar resultados serializables eliminando objetos no JSON
resultados_serializables = [
    {
        "nombre"          : r['nombre'],
        "n_chunks"        : r['n_chunks'],
        "sim_top1_global" : r['sim_top1_global'],
        "sim_media_global": r['sim_media_global'],
        "puntuacion"      : r['puntuacion'],
        "resultados"      : [
            {
                "pregunta"    : p['pregunta'],
                "nivel"       : p['nivel'],
                "ground_truth": p['ground_truth'],
                "sim_top1"    : p['sim_top1'],
                "sim_media"   : p['sim_media'],
            }
            for p in r['resultados']
        ],
    }
    for r in todos_resultados
]

output_json = {
    "fecha"              : datetime.now().strftime("%Y-%m-%d %H:%M"),
    "modelo_embeddings"  : MODELO_EMBEDDINGS,
    "top_k"              : TOP_K,
    "estrategia_optima"  : estrategia_optima['nombre'],
    "estrategias"        : resultados_serializables,
}

with open(RUTA_RESULTADOS, 'w', encoding='utf-8') as f:
    json.dump(output_json, f, ensure_ascii=False, indent=2)

print(f"Resultados persistidos en : {RUTA_RESULTADOS.name}")
print(f"Tamaño del archivo        : {RUTA_RESULTADOS.stat().st_size / 1024:.1f} KB")
print()
print("Evaluación comparativa completada.")
print("Siguiente paso: migrar a src/ (Paso 3 del proyecto)")

Resultados persistidos en : chunking_eval_results.json
Tamaño del archivo        : 7.3 KB

Evaluación comparativa completada.
Siguiente paso: migrar a src/ (Paso 3 del proyecto)
